# Homework 2. Biases in Verbalized Explanations, Chains-of-Thought, and Reasoning

Last time, we focused on LLMs' abilities to answer **factual** questions directly, i.e., the chances that they would answer explicitly "London" to the question "What's the capital of the UK?". However, sometimes we are interested in restricting the options for the answers an LLM can take (basically, a multiple-choice questions).

What's more important, we may be interested in **how** the model came to a particular answer, i.e., it's **reasoning** process. For example, take the LLM's abilities in maths: if we give a model an equation, we don't only expect an LLM to give the correct value of _x_, we also want it to be deduced according to the rules of maths, not to a lucky guess. This sub-area of model analysis is usually called **explainability**, because we expect a model to explain its answer and the way it came to it.

In this homework, we will tackle upon the problems that we can meet when we use multiple-choice questions when prompting LLMs with factual knowledge (Chapter 1), as well as the strong and weak points of the explainability techniques (Chapter 2).

## 0. Installation Etc.

### 0.1. Before We Start: How to Run the Code

This time, I would like you to try using GPU for the assignment. Since CPU/GPU distinction is not the main topic of the course, I'll not go into any details here; it will suffice to say that **these are two different types of hardware, and for maths- and matrix-heavy computations, GPUs are way more suitable**. For more details, you can watch [this video](https://www.youtube.com/watch?v=Axd50ew4pco$0). So GPUs should generate the answers way faster (and there is a risk that it wouldn't be able to generate the answers with CPU anyway, since it will be memory overload).

How can you run an assignment on GPU?
 - if you're using JupyterHub (default way): you need to **prepare the whole code** and **send the whole `.ipynb` at once to GPU**, and then get all outputs in a separate folder. It's a little inconvenient because you wouldn't be able to run separate cells; but it should not be a problem for this assignment (all the code for the LLM computation is ready). The instructions are provided [in this video](https://uzh.sharepoint.com/sites/UZHCloudServicesCommunity-JupyterHubTeachers/_layouts/15/stream.aspx?id=%2Fsites%2FUZHCloudServicesCommunity%2DJupyterHubTeachers%2FShared%20Documents%2FJupyterHub%20Batch%20Jobs%20%2D%20better%2Emp4&referrer=StreamWebApp%2EWeb&referrerScenario=AddressBarCopied%2Eview%2E6a644a9c%2D43f6%2D4fce%2Db449%2D2d378cabefce$0) (until 1:02). The more detailed instruction can be found at the [UZH JupyterHub page](https://jupyterhub.zi.uzh.ch/batch-jobs/$0). If I get enough messages that these instructions are not clear enough, I will record a more detailed video tutorial on that, so please keep me updated!
 - if you're using Colab Notebook, you can switch to a GPU by pressing "Runtime" (top-left menu) -> "Change Runtime Type" -> choose any option available to you that ends on "GPU" (I suggest using T4, not A100, because it's "cheaper" in terms of credits).
 - you can also run it on your local computer if you have a GPU, but I will leave it to your (+your coding agent😉) exploration.

 ### 0.2. Importing Libraries and LLM Modules

 - first, we'll import the necessary generic libraries.
 - second, you'll need to setup `transformers`-related libraries (such as `transformers` themselves, `datasets`, `torch` etc). I assume each of you may have the same problems as you had during HW1. So the best suggestion would be: **use the same solutions that you had for the previous assignment, to make your `transformers`-related imports work**. Also, **don't forget that there is a "Forum" page at our OLAT course, where student share their problems and solutions, including the setup obstacles**, so please check it from time to time to exchange your ideas.
 - finally, I have packed all code related to LLM answer generation into a separate `llm.py` file. You should see it in the folder. I did that for the sake of readability of the notebook, no significant changes compared to HW1 were made.

In [1]:
from pathlib import Path
import os

print("CWD:", Path.cwd())
print("HOME:", Path.home())

print("\nHOME contents:")
print(os.listdir(Path.home()))

print("\nHW2:")
print(os.listdir(Path.home() / "hw2"))

print("\nCredential exists:")
print((Path.home() / "hw2" / "credentials.json").exists())
print("Home/hw2/llm.py exists:", (Path.home() / "hw2" / "llm.py").exists())

CWD: /home/jovyan/hw2
HOME: /home/jovyan

HOME contents:
['.mypy_cache', 'HW1_clean-3.ipynb', 'Untitled.ipynb', '.jupyter_ystore.db-journal', '.ipynb_checkpoints', '.local', 'Model_Analysis_and_Interpretability_2026.git', '.jupyter_ystore.db', 'meme2.png', '.virtual_documents', 'jobs', '.fleet-home-meta', '.cache', 'mpararel_questions.json', 'HW1_clean.ipynb', 'hw2-2.ipynb', 'hw2', '.jupyter', 'Model_Analysis_and_Interpretability_2026', '.ipython', 'hw1_small', 'HW1_small.ipynb', '.config']

HW2:
['.ipynb_checkpoints', 'hw2_clean.ipynb', 'credentials.json']

Credential exists:
True


In [1]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.home() / "hw2",
    Path("/hw2"),
]

HW_DIR = next(
    (p.resolve() for p in candidates if (p / "llm.py").is_file()),
    None,
)

if HW_DIR is None:
    raise FileNotFoundError(
        "Could not find llm.py. "
        f"CWD={Path.cwd()}, HOME={Path.home()}"
    )

print("Importing llm.py from:", HW_DIR / "llm.py")
sys.path.insert(0, str(HW_DIR))

from llm import LLM

Importing llm.py from: /home/jovyan/hw2/llm.py


In [2]:
# Step 1: Importing libraries.
import time
import json
import re
import random
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter, defaultdict
from typing import Union, List, Dict, Any, Optional
import copy
import pandas as pd
import seaborn as sns
from tqdm import tqdm


In [3]:
# huggingface login to download the models
from huggingface_hub import notebook_login
notebook_login()

In [4]:
import os
os.environ['LD_LIBRARY_PATH'] = "/opt/conda/lib/python3.13/site-packages/nvidia/cu13/lib:" + os.environ.get('LD_LIBRARY_PATH', '')
import torch
print(torch.__version__)

2.9.1+cu130


In [5]:
# Step 2: Importing transformers-related packages
# TODO-before-GPU: try using the same pieces of code that worked with HW1
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset, load_dataset


In [6]:
# Step 3: importing the LLM-related module, and initializing two LLMs.
from llm import LLM

In [7]:
llm = LLM("google/gemma-3-1b-it", quantized=False, token_path=Path.home() / "hw2" / "credentials.json")


Loading tokenizer for google/gemma-3-1b-it...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Tokenizer loaded successfully (no token required).
CPU device detected. Using CPU.
Loading model with bfloat16...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model and tokenizer loaded successfully!


In the end of the assignment, we'd also work with another LLM, [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B$0). Let's initialize it here to see if anything is wrong early on.

Loading tokenizer for Qwen/Qwen3-0.6B...
Tokenizer loaded successfully (no token required).
CPU device detected. Using CPU.
Loading model with bfloat16...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Model and tokenizer loaded successfully!


### 0.3. Parts of your assignment

1. `# TODO-before-GPU` - make sure you fill in everything before sending it to GPU. Without that, your code will crash.

2. `# TODO-assignment` - this is the "actual" part of your assignment. I tried to make it in such a way that you will not need to iteratively switch to GPU for that part; so you can run the GPU first, and then work at JupyterHub's CPU with the outputs from GPU that you just got.

(show what parts need to be submitted)

# Chapter 1: More on prompt brittleness; the role of few-shot prompting

##### Multiple-Choice Prompts

In HW1, we tried to analyze the LLM's ability to extract the knowledge of particular facts in two ways:
 - letting it continue the sentence generation (`London is the capital of...`), we called it `no-chat` setup.
 - asking it directly and expecting it to give the answer (`Of which country is London the capital?`), this we called `chat` setup.

We saw that model's answers are unstable, or **brittle**, because of different factors:
 - whether we prompt the model in a `no-chat` or `chat` manner;
 - whether, for the fact "X-relation-Y", we ask a model about X (given Y) or Y (given X) - **reversal curse**;
 - depending on a particular formulation of the relation (`Le Corbusier was born in...` or `Le Corbusier's birthplace is...`) - **knowledge consistency**.

To solve that, researchers tried to come up with other ways of model prompting. One of which is essentially repeating the multiple-choice answers that we are given in school. For example, the question about London and the UK will be formulated as:

```
Of what country is London the capital?
(A) The US
(B) Russia
(C) The UK
(D) France
```
The LLM then will be expected to output only the letter (in this case, `(C)`).

#### Exercise 1.1. Answer the questions.

Why might researchers have turned to multiple-choice prompting, and how does it improve on the direct question-answering pairs we analyzed in HW1?

1. Consider the `p.1` example in the code box below. We prompt a model to complete `London is the capital of` 8 times, using slightly different temperatures*. In HW1, we evaluated answers by searching for the true value, `the UK`, using `regex`. What is one disadvantage of this approach compared to multiple-choice prompting? [1 point]
2. LLMs process text as sequences of **subword tokens**—atomic character sequences (for details, watch [this video](https://www.youtube.com/watch?v=id8guuz8U8M$0)). In the next code box (`p.2`), we examine how our LLM tokenizes different country names, including variants of the same name (e.g., "The UK" vs. "The United Kingdom...") and different names (e.g., "France" or "Ireland"). What risk does this pose when evaluating fully generated answers (as in HW1), and how can multiple-choice prompting mitigate it? [1 point]
3. Suppose we ask a model to solve a math equation, where $\frac{2\sqrt(3)\pi}{3}$ is a true answer. What problem might arise if the model can generate the answer in any format it chooses? (Hint: This is also partly related to tokenization.) [1 point]
4. Give one additional argument for and one against using multiple-choice answers to analyze LLM knowledge. [2 points]

\* Roughly speaking, temperature controls the "creativity" or randomness of a model's continuations. However, the difference between 0.95 and 1.05 is small. For this assignment, treat temperature as a way to generate different versions of an answer to the same question.

In [9]:
# p.1: sentence continuation for `London is a capital of...`

input = ("London is the capital of ")
for t in np.linspace(0.95, 1.05, num=8):
    output = llm.generate_response(input, chat_setup=False, only_outputs=True, max_new_tokens=25, temperature=t)
    print(f'temperature: {t}, output: ...{output}')

temperature: 0.95, output: ...
 
  A) Scotland
  B) Wales
  C) England
  D) Ireland

The answer
temperature: 0.9642857142857143, output: ...
    United Kingdom,
    England and Scotland.

temperature: 0.9785714285714285, output: ...
(A) England
(B) Scotland
(C) Wales
(D) Ireland
The correct answer is
temperature: 0.9928571428571429, output: ...
 
   
A) England
B) Scotland
C) Wales
D) Ireland 

The correct answer
temperature: 1.0071428571428571, output: ...
(a) England
(b) Wales
(c) Scotland
(d) Ireland

The correct answer is
temperature: 1.0214285714285714, output: ...
A. France
B. Spain
C. Germany
D. Italy

The correct answer is C. Germany.
temperature: 1.0357142857142858, output: ...
(1) England
(2) France
(3) Scotland
(4) Wales

The correct answer is
temperature: 1.05, output: ...
A. Scotland
B. England
C. Wales
D. Ireland

The correct answer is B. England.


In [10]:
# p.2: tokenization of different proper names
for alias in ['the UK', 'Great Britain', 'the United Kingdom of Great Britain and Northern Ireland', 'France', 'Ireland']:
  tokens = llm.tokenizer.encode(alias)
  print(f'Proper Name: {alias}, number of tokens: {len(tokens)}')


Proper Name: the UK, number of tokens: 3
Proper Name: Great Britain, number of tokens: 3
Proper Name: the United Kingdom of Great Britain and Northern Ireland, number of tokens: 10
Proper Name: France, number of tokens: 2
Proper Name: Ireland, number of tokens: 2


In any case, multiple-choice questions (also known as **close-form questionnaires**, as opposed to **open-form** question-answering where a model can generate any text) are a popular practice in NLP nowadays.

Let's work with an LLM evaluation dataset that is formed exactly in this way - [MMLU](https://arxiv.org/pdf/2009.03300$0): it provides questions related to different subjects, and four options for each question.

In [11]:
ds_mmlu = load_dataset('cais/mmlu', 'all')

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.50MB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

all/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  408kB            

...

all/auxiliary_train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Let's take a look at it:
 - `question` is a question formulated in a `chat` style (from the last assignment)
 - `subject` is the topic of a question
 - `choices` is a list of 4 answer options
 - `answer` is an idx of the list with the correct answer (0 to 3)

In [12]:
pd.DataFrame(ds_mmlu['test']).sample(5)

,question,subject,choices,answer
3354,This question refers to the following informat...,high_school_european_history,"[the Aristotelian tradition, the natural magic...",2
1687,"According to Coulomb, a pair of charged partic...",conceptual_physics,"[twice as strong, four times as strong, half a...",1
8087,"In a holistic evaluation of student essays, ev...",miscellaneous,[number and variety of errors made by each stu...,3
12372,During a clinical study examining the effects ...,professional_medicine,"[Pulse pressure, Stroke volume, Systolic blood...",3
2305,A tournament has 393 players with 3 players on...,elementary_mathematics,"[393 + 3, 393 – 3, 393 × 3, 393 ÷ 3]",3


Below, you see the list of subjects in the dataset. To continue with the assignment, you need to choose a subject. You are free to choose the subject of your interest. I like religious studies, so I took the `world_religions` in the field below and created a `subset` variable with questions related only to that.

**NB: Please don't choose the maths-related topics! This will be discovered in Chapter 2.**

In [13]:
pd.DataFrame(ds_mmlu['test'])['subject'].unique()

<ArrowStringArray>
[                   'abstract_algebra',                             'anatomy',
                           'astronomy',                     'business_ethics',
                  'clinical_knowledge',                     'college_biology',
                   'college_chemistry',            'college_computer_science',
                 'college_mathematics',                    'college_medicine',
                     'college_physics',                   'computer_security',
                  'conceptual_physics',                        'econometrics',
              'electrical_engineering',              'elementary_mathematics',
                        'formal_logic',                        'global_facts',
                 'high_school_biology',               'high_school_chemistry',
        'high_school_computer_science',        'high_school_european_history',
               'high_school_geography', 'high_school_government_and_politics',
          'high_school_macroecono

In [14]:
# TODO-before-GPU: choose the subject name and put it into the `subject` variable
subject = 'world_religions'
subset = ds_mmlu.filter(lambda x: x['subject'] == subject)

Filter:   0%|          | 0/14042 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1531 [00:00<?, ? examples/s]

Filter:   0%|          | 0/285 [00:00<?, ? examples/s]

Filter:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [15]:
subset['test'][12]

{'question': 'What is the communal meal offered at the place of worship called in Sikhism?',
 'subject': 'world_religions',
 'choices': ['Sangat', 'Langar', 'Gurdwara', 'Panth'],
 'answer': 1}

We need to format the data for the model by converting it into continuous text and tracking the letters assigned to correct answers (instead of their list indices).

We'll use the `create_text_from_datapoint` function for this.

The function takes a single dataset question (`datapoint`) and converts it into continuous text. It also accepts the following arguments:

- `perturb_labels`: LLMs can be sensitive to the order of answer options, as we'll see in the code below. This parameter controls how options are reordered or modified (`None` leaves them unchanged; other values will be explained later).
- `include_answer`: whether to include the correct answer's letter. We'll use this option when providing the model with example questions and answers; set it to `True` for such samples.
- `include_options`: set to `False` to use the simple question-answering setup from HW1, which we'll need for the experiments in Chapter 2.

The function returns:

- `text`: the formatted question text.
- `correct_letter`: the letter assigned to the correct answer.
- `distractor_letter`: in some experiments (Chapter 2), we'll test whether an LLM can be misled by intentionally highlighting a wrong answer option. This variable stores the letter of a **randomly chosen wrong answer option**.
- `correct_choice`: the exact textual answer (e.g., `the UK`), which we'll also use in the experiments in Chapter 2.

In [16]:
def create_text_from_datapoint(datapoint: dict, perturb_labels: str = None, include_answer: bool = False, include_options: bool = True) -> tuple[str, str, str, Any]:
    '''
    Converts a dataset datapoint into a formatted question string with multiple-choice options.
    Supports label perturbation modes for studying model bias and behavior.

    Parameters:
      - datapoint: dict containing 'question', 'choices', and 'answer' (or 'answer_value')
      - perturb_labels: whether to shuffle the answer options {None, 'random', 'first', 'inject'}
          - None: the order remains as in the dataset
          - 'random': the order is shuffled randomly
          - 'first': the correct answer is moved to the first position (index 0, corresponds to letter 'A')
          - 'inject': the correct answer is provided with HTML-like tags (e.g., <answer>...</answer>)
      - include_answer: If True, appends the correct answer letter to the prompt
      - include_options: If True, includes the verbalized multi-choice option list

    Returns:
      - text: The formatted question/prompt string
      - correct_letter: The letter (A, B, C, D...) corresponding to the correct answer in the final options list
      - distractor_letter: The letter corresponding to the target wrong answer (for 'first' or 'inject' modes), otherwise 'F'
      - correct_value: The direct textual or numeric value of the correct answer option
    '''
    # 1. Extract the question and options
    question = datapoint['question']
    choices = list(datapoint['choices'])

    # Identify the correct choice using either the answer index or the direct correct answer value
    correct_choice = choices[datapoint['answer']]

    # 2. Initialize helper variables
    distractor_letter = 'F'
    target_wrong_choice = None

    # 3. Apply the label perturbation flow step-by-step
    if perturb_labels is None:
        # Standard multi-choice: options remain in their original order
        final_choices = list(choices)

    elif perturb_labels == 'random':
        # Shuffle the choices list randomly
        final_choices = list(choices)
        random.shuffle(final_choices)

    elif perturb_labels == 'first':
        # Move the correct answer to the first position (index 0, which corresponds to letter 'A')
        wrong_choices = [c for c in choices if c != correct_choice]
        final_choices = [correct_choice] + wrong_choices
        distractor_letter = 'A'  # The target answer is intentionally positioned at A

    elif perturb_labels == 'inject':
        # Select one random wrong choice to wrap in the target xml tag
        final_choices = list(choices)
        wrong_choices = [c for c in choices if c != correct_choice]
        if wrong_choices:
            target_wrong_choice = random.choice(wrong_choices)
        else:
            target_wrong_choice = None

    else:
        raise ValueError(f"Unknown perturbation mode: {perturb_labels}")

    # 4. Find the final correct answer letter
    correct_idx = final_choices.index(correct_choice)
    correct_letter = chr(65 + correct_idx)  # Maps 0 -> 'A', 1 -> 'B', etc.

    # 5. Build and format the verbalized choices list
    formatted_options_list = []
    for i, choice in enumerate(final_choices):
        letter = chr(65 + i)

        # If we are in 'inject' mode and this choice matches our designated target wrong choice
        if perturb_labels == 'inject' and choice == target_wrong_choice:
            formatted_options_list.append(f"- <answer>({letter}) {choice}</answer>")
            distractor_letter = letter  # Save the letter that was injected with tags
        else:
            formatted_options_list.append(f"- ({letter}) {choice}")

    formatted_options = "\n".join(formatted_options_list)

    # 6. Construct the final output prompt text
    if include_options:
        text = f"Question: {question}\nAnswer options:\n{formatted_options}\nCorrect answer: "
    else:
        text = f"Question: {question}\n"

    if include_answer:
        text += f"({correct_letter})"

    return text, correct_letter, distractor_letter, correct_choice

Here's an example of the function's output:

In [17]:
sample_dp = subset['test'][67]
print('input:')
print(sample_dp)

input:
{'question': 'How many female renouncers were under the tutelage of Chandanbala?', 'subject': 'world_religions', 'choices': ['36,000', '12,000', '24,000', '7,000'], 'answer': 0}


In [18]:
text, corr_letter, dist_letter, corr_choice = create_text_from_datapoint(sample_dp, perturb_labels=None, include_answer=False)
print('output:')
print(f'prompt text:\n{text}\n===========')
print(f'correct_letter: {corr_letter}\n===========')
print(f'distractor_letter: {dist_letter}\n===========')
print(f'correct_choice: {corr_choice}\n===========')


output:
prompt text:
Question: How many female renouncers were under the tutelage of Chandanbala?
Answer options:
- (A) 36,000
- (B) 12,000
- (C) 24,000
- (D) 7,000
Correct answer: 
correct_letter: A
distractor_letter: F
correct_choice: 36,000


In [19]:
subset['validation']

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 19
})

##### Few-Shot Examples

Another popular LLM prompting technique is **few-shot prompting**. It addresses the problem of LLMs not always "knowing" what we expect from them. For example, if we prompt a model with `Le Corbusier was born in`, it might interpret this as a question about the birth year and answer `1887`, or provide a country name, such as `Switzerland`, instead of a birthplace. However, [empirical evidence](https://arxiv.org/pdf/2005.14165$0) suggests that providing several examples can help narrow down the expected answer format.

For instance, if we prompt an LLM with the following text:

```
Leo Tolstoy was born in Yasnaya Polyana.
Marilyn Monroe was born in Los Angeles.
Cao Xueqin was born in Nanjing.
Le Corbusier was born in...
```

The model will likely provide a location (rather than a year or country), making its answer easier to evaluate.

Providing a model with several question-answer examples is called **few-shot prompting**. We can vary the number of examples: prompts with 4 examples are called **4-shot prompts**, those with 16 examples are **16-shot prompts**, and so on. Conversely, prompting a model without examples, as in HW1, is called **zero-shot prompting**.

Researchers have studied the optimal number of examples (e.g., [here](https://aclanthology.org/2024.findings-naacl.115.pdf$0)), but there appears to be no universally optimal choice. Instead, we'll experiment with different numbers of examples ourselves.

We will make a function that creates few-shot examples for an LLM - `create_few_shot_prompt_base`. It will take on:
- `few_shot_examples`: a pool of the data points from which it will create the examples. We will use the `validation` split of the MMLU dataset of a given subject.
- `perturb_labels`: same as in `create_text_from_datapoint`, defines if we want to rearrange the answer options in a particular way.
- `n` - the number of shots.

In [20]:
def create_few_shot_prompt_base(few_shot_examples, perturb_labels: str = None, n: int = 4) -> str:
    few_shot_prompt_base = ""

    # Handle HF Dataset or Dataset slice (dict of lists) or list of dicts
    if hasattr(few_shot_examples, 'num_rows'):
        num_examples = min(few_shot_examples.num_rows, n)
        examples_list = [few_shot_examples[i] for i in range(num_examples)]
    elif isinstance(few_shot_examples, dict):
        keys = few_shot_examples.keys()
        num_examples = min(len(next(iter(few_shot_examples.values()))), n)
        examples_list = [
            {key: few_shot_examples[key][i] for key in keys}
            for i in range(num_examples)
        ]
    else:
        examples_list = few_shot_examples[:n]

    for i, example in enumerate(examples_list):
        verbalized_example, _, _, _ = create_text_from_datapoint(example, perturb_labels=perturb_labels, include_answer=True)
        few_shot_prompt_base += f"{verbalized_example}\n\n"
    return few_shot_prompt_base

Here's an example of that function, for four-shot prompt with random perturbation of the answer options. As you see, all answers are explicitly provided.

In [21]:
perturb_random_4shot = create_few_shot_prompt_base(subset['validation'], n=4, perturb_labels='random')
print(perturb_random_4shot)

Question: Which of the following is NOT a category of Shari'ah law?
Answer options:
- (A) Commendable
- (B) Neutral
- (C) Criminal
- (D) Unlawful
Correct answer: (C)

Question:  Which Guru ordered the building of the Darbar Sahib, also known as the Divine Court?
Answer options:
- (A) Guru Amar Das
- (B) Guru Arjan
- (C) Guru Nanak
- (D) Guru Ram Das
Correct answer: (B)

Question:  Yamago Soko created the "way of the warrior," known as which of the following?
Answer options:
- (A) Gomadaki
- (B) Samurai
- (C) Bushido
- (D) Nembutsu
Correct answer: (C)

Question: Which of the following was part of the role of a deaconess?
Answer options:
- (A) All of the above
- (B) Praying for the suffering
- (C) Ministering to the sick
- (D) Preparing women for baptism
Correct answer: (A)




And this is an example of the 4-shot prompt where all answers are perturbed in a way that the correct option is always (A). This will be needed for our experiment below.

In [22]:
perturb_first_4shot = create_few_shot_prompt_base(subset['validation'], n=4, perturb_labels='first')
print(perturb_first_4shot)

Question: Which of the following is NOT a category of Shari'ah law?
Answer options:
- (A) Criminal
- (B) Neutral
- (C) Unlawful
- (D) Commendable
Correct answer: (A)

Question:  Which Guru ordered the building of the Darbar Sahib, also known as the Divine Court?
Answer options:
- (A) Guru Arjan
- (B) Guru Nanak
- (C) Guru Ram Das
- (D) Guru Amar Das
Correct answer: (A)

Question:  Yamago Soko created the "way of the warrior," known as which of the following?
Answer options:
- (A) Bushido
- (B) Samurai
- (C) Nembutsu
- (D) Gomadaki
Correct answer: (A)

Question: Which of the following was part of the role of a deaconess?
Answer options:
- (A) All of the above
- (B) Ministering to the sick
- (C) Preparing women for baptism
- (D) Praying for the suffering
Correct answer: (A)




Let's first examine how the number of few-shot examples affects a model's performance on multiple-choice question answering. We'll use three functions, starting with `run_experiment`, which generates the model's answers. It accepts the following arguments:

- `llm`: the LLM being evaluated.
- `test_examples`: the dataset used for evaluation (the `test` split of `subset`).
- `few_shot_examples`: the output of the function defined above, containing formatted examples.
- `general_instruction` and `final_instruction`: additional instructions placed before the examples or question, respectively, to guide the model's response format.
- `perturb_labels`: reordering of answer options, same as above.
- `include_options`: used in Chapter 2; if `False`, the multiple-choice options are omitted.
- `chat_setup`, `batch_size`, `only_outputs`, `max_new_tokens`: the same parameters as in `llm.generate_batch_response`.

The function returns a `pandas` DataFrame containing the prompts, model responses, correct answer letters and values, and distractor labels.

In [1]:
def run_experiment(
    llm: LLM,
    test_examples: Union[List[dict], dict, Any],
    few_shot_examples: Optional[str] = None,
    general_instruction: Optional[str] = None,
    final_instruction: Optional[str] = None,
    perturb_labels: Optional[str] = None,
    include_options: Optional[bool] = True,
    chat_setup: bool = True,
    batch_size: int = 4,
    only_outputs: bool = True,
    max_new_tokens: int = 15
) -> pd.DataFrame:
    """
    Runs an LLM evaluation experiment over a batch of test examples, supporting
    multi-choice options formatting, few-shot prompting, and label perturbations.

    Parameters:
        - llm: LLM
            The language model object used to generate responses.
        - test_examples: Union[List[dict], dict, Any]
            The dataset split containing evaluation questions (e.g., subset['test']).
        - few_shot_examples: Optional[str], default=None
            Formatted string containing few-shot examples to prepend to each prompt.
        - general_instruction: Optional[str], default=None
            Instruction added at the very beginning of the prompt (e.g., system-level instructions).
        - final_instruction: Optional[str], default=None
            Formatting instructions added right before the question text (e.g., output constraint formats).
        - perturb_labels: Optional[str], default=None
            Reordering/perturbation strategy for options. Supported modes: {None, 'random', 'first', 'inject'}.
        - include_options: Optional[bool], default=True
            If True, includes verbalized multiple-choice choices in the prompt; if False, omits options.
        - chat_setup: bool, default=True
            Whether to apply the chat template formatting wrapper before generating outputs.
        - batch_size: int, default=50
            The batch size of prompts sent concurrently to the model.
        - only_outputs: bool, default=True
            If True, only returns the model's generated text, removing the input prompt prefix.
        - max_new_tokens: int, default=15
            The maximum limit of new tokens the model can generate for each prompt.

    Returns:
        - df_results: pd.DataFrame
            DataFrame containing the evaluation outputs with the following columns:
            - 'prompt': The complete prompt text sent to the model.
            - 'response': The raw response generated by the model.
            - 'correct_label': The ground-truth correct option letter (A, B, C, etc.).
            - 'correct_value': The literal correct textual/numeric value of the answer.
            - 'distractor_label': The target letter containing the adversarial or first-placed wrong answer, or 'F'.
    """

    # Determine the number of examples in test_examples
    if hasattr(test_examples, 'num_rows'):
        examples_list = [test_examples[i] for i in range(test_examples.num_rows)]
    elif isinstance(test_examples, dict):
        keys = test_examples.keys()
        num_examples = len(next(iter(test_examples.values())))
        examples_list = [
            {key: test_examples[key][i] for key in keys}
            for i in range(num_examples)
        ]
    else:
        examples_list = test_examples

    prompts = []
    correct_labels = []
    correct_values = []
    distractor_labels = []

    for datapoint in examples_list:
        prompt_parts = []
        if general_instruction:
            prompt_parts.append(general_instruction)
        if few_shot_examples:
            prompt_parts.append(few_shot_examples)

        # Standard multi-choice flow
        question_text, correct_letter, distractor_letter, correct_choice = create_text_from_datapoint(
            datapoint,
            perturb_labels=perturb_labels,
            include_answer=False,
            include_options=include_options
        )
        if final_instruction:
            prompt_parts.append(final_instruction)

        prompt_parts.append(question_text)

        # Combine instructions, examples, and the question
        full_prompt = "\n".join([part for part in prompt_parts if part])
        prompts.append(full_prompt)
        correct_labels.append(correct_letter)
        correct_values.append(correct_choice)
        distractor_labels.append(distractor_letter)

    # Generate responses using batch processing
    responses = llm.generate_batch_response(
        prompts,
        batch_size=batch_size,
        max_new_tokens=max_new_tokens,
        chat_setup=chat_setup,
        only_outputs=only_outputs
    )

    # Construct the result DataFrame with extended metadata
    df_results = pd.DataFrame({
        'prompt': prompts,
        'response': responses,
        'correct_label': correct_labels,
        'correct_value': correct_values,
        'distractor_label': distractor_labels
    })
    return df_results

NameError: name 'LLM' is not defined

Let's see an example of this function's output:

In [25]:
df_output = run_experiment(
    llm=llm,
    test_examples=subset['test'],
    few_shot_examples=perturb_random_4shot,
    general_instruction="Please select the correct option to answer the question.",
    batch_size=8)

processing 0/1...


In [2]:
pd.options.display.max_colwidth = 250

NameError: name 'pd' is not defined

In [ ]:
df_output.head(20)

The second function, `evaluate_answer_labels`, will evaluate the LLM answers against the true labels. It will check whether there is the title-case letter in parentheses, and will compare it against the `correct_label` column. If it matches, it will return `True`.

The argument `exclude_reasoning_tokens` will be needed later, in Chapter 2.

In [28]:
def evaluate_answer_labels(answers: pd.DataFrame, exclude_reasoning_tokens: bool = False) -> pd.DataFrame:
    # Function to extract an uppercase letter in parentheses (A-D) from response
    def extract_letter(response):
        if not isinstance(response, str):
            return 'F'

        text_to_search = response
        if exclude_reasoning_tokens:
            # Greedy match to remove everything from the first <think> to the last </think>
            # This also handles malformed headers or multiple <think> tags
            text_to_search = re.sub(r'<think>.*</think>', '', response, flags=re.DOTALL)

        # Look for patterns like (A), (B), (C), (D) in the response text
        match = re.search(r'\(([A-D])\)', text_to_search)
        if match:
            return match.group(1)
        return 'F'

    # Create a copy to prevent SettingWithCopyWarning
    df = answers.copy()
    df['response_clean'] = df['response'].apply(extract_letter)
    df['correct'] = df['response_clean'] == df['correct_label']
    return df

The third function, `compare_few_shot_examples`, compares model performance across different numbers of few-shot examples. It accepts the following arguments:

- `dataset`: the dataset to use, in our case `subset`.
- `llm`: the LLM to evaluate.
- `perturbation`: same as `perturb_options`.
- `n`: a list of values for *n*-shot prompts.
- `tofile`: whether to save the evaluation results to a file.

The function generates a set of graphs, one for each *n*, showing the distribution of correct answers across option letters, the model's predictions for each letter, and how many predictions were correct. It also saves the model outputs and evaluation results to a `.tsv` file in the batch output folder, so that you can work with it after GPU finishes the work.

In [30]:
def compare_few_shot_examples(dataset, llm, perturbation='random', n=[1, 4, 8, 16], tofile=True):
    """
    Parameters:
        - dataset: Dataset
            The evaluation dataset split containing 'validation' and 'test' splits.
        - llm: LLM
            The language model instance used to generate predictions.
        - perturbation: Optional[str], default='random'
            The option reordering/perturbation strategy used for few-shot prompt construction.
        - n: List[int], default=[1, 4, 8, 16]
            List of integers representing the number of few-shot examples (n-shot) to evaluate.
        - tofile: bool, default=True
            If True, saves the combined evaluation results to a tab-separated (.tsv) file.

    Returns:
            Displays the matplotlib label distribution subplots directly in the notebook.
    """
    shot_configs = []
    for n_shot in n:
        if n_shot == 0:
            few_shot_prompt_base = ""
        else:
            few_shot_prompt_base = create_few_shot_prompt_base(dataset['validation'], n=n_shot, perturb_labels=perturbation)
        shot_configs.append((n_shot, f"{n_shot}-Shot", few_shot_prompt_base))

    evaluated_results = {}
    all_dfs = []

    # Run evaluation loop
    for n_shot, name, prompt_base in shot_configs:
        print(f"Evaluating {name}...")
        raw_df = run_experiment(
            llm=llm,
            test_examples=dataset['test'], 
            few_shot_examples=prompt_base if n_shot > 0 else None,
            general_instruction="Please select the correct option to answer the question.",
            perturb_labels='random',
            chat_setup=False,
            batch_size=4,
            only_outputs=True,
            max_new_tokens=5
        )
        eval_df = evaluate_answer_labels(raw_df)
        evaluated_results[name] = eval_df

        if tofile:
            df_copy = eval_df.copy()
            df_copy['n'] = n_shot
            all_dfs.append(df_copy)

    if tofile and all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        combined_df.to_csv(f'few_shot_{",".join([str(n_) for n_ in n])}_comparison_{perturbation}_results.tsv', sep='\t', index=False)
        print("Saved combined results to 'few_shot_comparison_results.tsv'")

    # Visualize distributions in scalable subplots (2 columns)
    letters = ['A', 'B', 'C', 'D']
    x = np.arange(len(letters))
    width = 0.35

    num_plots = len(evaluated_results)
    ncols = 2
    nrows = (num_plots + 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 6 * nrows), sharey=True, squeeze=False)
    axes_flat = axes.flatten()

    for i, (name, df_eval) in enumerate(evaluated_results.items()):
        ax = axes_flat[i]

        # Calculate counts and accuracy
        correct_preds = []
        incorrect_preds = []
        for letter in letters:
            subset_letter = df_eval[df_eval['response_clean'] == letter]
            correct_count = subset_letter['correct'].sum()
            incorrect_count = len(subset_letter) - correct_count
            correct_preds.append(correct_count)
            incorrect_preds.append(incorrect_count)

        correct_preds = np.array(correct_preds)
        incorrect_preds = np.array(incorrect_preds)
        correct_counts = df_eval['correct_label'].value_counts().reindex(letters, fill_value=0)


        # Plot stacked predictions
        ax.bar(x - width/2, correct_preds, width, label='Model Outputs (Correct)',
              color='skyblue', edgecolor='black', hatch='//')
        ax.bar(x - width/2, incorrect_preds, width, bottom=correct_preds,
              label='Model Outputs (Incorrect)', color='lightsteelblue', edgecolor='black')

        # Plot ground truth
        ax.bar(x + width/2, correct_counts, width, label='Ground Truth',
              color='salmon', edgecolor='black')

        ax.set_title(f"{name}")
        ax.set_xticks(x)
        ax.set_xticklabels(letters)
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        if i == 0:
            ax.legend()
        if i % ncols == 0:
            ax.set_ylabel('Counts')

    # Hide unused subplots if any
    for j in range(i + 1, len(axes_flat)):
        fig.delaxes(axes_flat[j])

    fig.suptitle(f'Label Distributions for Few-Shot Examples, {perturbation + " Answer Perturbation" if perturbation is not None else ""}', fontsize=16, y=1.02)
    fig.tight_layout()
    plt.show()

In [ ]:
compare_few_shot_examples(subset, llm, perturbation=None, n=[1, 4, 8, 16], tofile=True)

The results should be saved in a `.tsv` file named `few_shot_{...}_comparison_None_results.tsv` (where {...} stands for the exact sizes of few-shot samples, and "None" stands for the `perturbation` value). You can open it with `pandas` as below, or you can download the dataset and look at it yourself with, say, Excel or other table editor ("Data" -> "Get Data" -> "Text/CSV" -> select the downloaded file).

In [ ]:
df = pd.read_csv('few_shot_1,4,8,16_comparison_None_results.tsv', sep='\t')
df.tail(10)

Few-shot examples seem to help LLMs retrieve knowledge, at least to some extent.

However, we must be careful when constructing these examples. Imagine the following scenario:

- We create few-shot examples manually.
- For each question, we note the correct answer first and then quickly come up with three incorrect options.
- We forget to shuffle the options, even though the test dataset is balanced across correct-answer letters.
- As a result, the correct answer is always the first option in the few-shot examples, but not necessarily in the test data.

Would this be a problem for a human taking a test? Probably not, although the examples might seem unusual. But could this pattern influence an LLM's predictions?

Let's find out using the `perturbation="first"` option in our experiment. It simulates this scenario by moving the correct answer to the first position in every few-shot example.

In [ ]:
compare_few_shot_examples(subset, llm, perturbation="first", n=[1, 4, 8, 16], tofile=True)

#### Exercise 1.2. Answer the questions:

1. Include the zero-shot scenario in both experiments (the relevant fields are provided below). You only need to change one element. Was few-shot prompting generally useful for evaluating MMLU on your subject, or we could do well with zero-shot prompting? [1 point]
2. Compare the distributions of the correct answers and all LLM-generated answers, both with no perturbation and with `"first"` perturbation. What common trends and differences do you observe? [1 point]
3. The graphs show the distribution of correct answers across labels, but not overall model performance. Open the generated `.tsv` files, group the results by *n*, and calculate accuracy (the percentage of correct answers) for each *n*. Present your results in a plot or table. Since accuracy trends may vary by subject, describe the trend for yours. Do you notice anything unusual/notable? [2 points]
4. I suggested that humans should not be sensitive to label order in few-shot examples or multiple-choice questions. Is this true? Give an anecdotal example or cite research showing that humans can also exhibit biases when answering multiple-choice questions. [1 point]
5. Open the saved `.tsv` file and, for the `"first"` perturbation, examine the data points where `n = 16` (i.e., the 16-shot examples). The bar charts show a strong bias toward option A, yet the models sometimes correctly select other options. Manually analyze these questions and answers. Do you notice any common patterns among the correctly answered non-A options? Consider question complexity, topic, the relevance of incorrect options, or other possible factors. [4 points]

In [ ]:
# TODO-before-GPU: change one small part in the code to include zero-shot prompting
compare_few_shot_examples(subset, llm, perturbation=None, n=[1, 4, 8, 16], tofile=True)

In [ ]:
# TODO-assignment: Exercise 1.2.3

Bonus [meme](https://www.instagram.com/reel/Dd77hleCbFW/?srtk=MXBseW84NDNvdDFuMQ%3D%3D$0) (relevant for this chapter) for those who got this far!

# 2. Mathematical Reasoning and Explainability

Consider these two questions:

1. What is $\frac{\sqrt625}{\frac{1}{2}} + \frac{3!}{\sqrt[3]{-8}}$?
2. What is the capital of Kiribati?

Both might be considered moderately difficult (unless you're from Kiribati), but they differ in one important way: the first can be broken down into steps (e.g., calculating the roots, performing the divisions, and adding the results), while the second cannot be decomposed in the same way.

For both humans and LLMs, simply getting the answer `47` to the first question may not be enough. We'd also like to understand how the answer was obtained—in other words, the **reasoning** involved, (in this case, **mathematical reasoning**). This is the focus of Chapter 2.

We'll continue using the MMLU dataset, which includes four mathematics-related subsets:

- `elementary_mathematics`
- `college_mathematics`
- `high_school_mathematics`
- `abstract_algebra`

You can choose any of these subsets. The function below selects the corresponding dataset, converts answer values to numeric types, and filters out options containing extensive markup, retaining only integer or floating-point answers.

In [ ]:
def get_maths_subset(dataset, maths_subject):

    def transform_list_to_numbers(lst):
        # lst = ast.literal_eval(lst)
        result = []
        for string in lst:
            try:
                result.append(int(string))
            except ValueError:
                try:
                    result.append(float(string))
                except ValueError:
                    return None
        return result

    subset = dataset.filter(lambda example: example['subject'] == maths_subject)
    # transforming to format the options and answers.
    # we filter out all data points which have other answer options than ints or floats.
    df = pd.DataFrame(subset['test'])
    df['choices_formatted'] = df['choices'].apply(transform_list_to_numbers)
    subset = df[~df['choices_formatted'].isna()]
#    subset['answer_value'] = subset.apply(lambda row: row['choices_formatted'][row['answer']], axis=1)
    subset = subset.drop(columns=['choices'])
    subset = subset.rename(columns={'choices_formatted': 'choices'})
    subset = Dataset.from_pandas(subset)
    return subset



In [ ]:
# TODO-before-GPU: choose a maths subject
subject = 'high_school_mathematics'
maths_subset = get_maths_subset(ds_mmlu, subject)

First, we'll compare the LLM's performance with and without answer options. We'll run two experiments:

- In `maths_0shot_value`, the model won't see the answer options (`include_options=False` in `create_text_from_datapoint`). Instead, we'll ask it to generate the exact numerical answer in the format `Answer: **{number}**`. The `evaluate_maths_answers` function below will compare its responses with the true values in the `correct_value` column.
- In `maths_0shot_label`, the model will see the answer options and must respond with the corresponding letter in the format `({letter})`. We'll evaluate its responses against `correct_label`.

Each experiment's results will be saved to a `{experiment_name}.tsv` file.

In [ ]:
def evaluate_maths_answers(answers: pd.DataFrame) -> pd.DataFrame:
    # Function to extract numbers from patterns like Answer: **10** or Answer: **332.74**
    def extract_maths_answer(response):
        if not isinstance(response, str):
            return None
        # Look for pattern 'Answer: **number**' with optional decimals
        match = re.search(r'Answer:\s*\*\*(-?\d+(?:\.\d+)?)\*\*', response)
        if match:
            # Return as float for robust numerical matching
            return float(match.group(1))
        return None

    # Create a copy to prevent SettingWithCopyWarning
    df = answers.copy()
    df['response_clean'] = df['response'].apply(extract_maths_answer)

    # Check correctness
    df['correct'] = df['response_clean'] == df['correct_value']
    return df

In [ ]:

maths_0shot_value_df = run_experiment(
    llm=llm,
    test_examples=maths_subset,
    few_shot_examples=None,
    general_instruction="Please answer the question. Format it as 'Answer: ' and the following number in bold font, for example, 'Answer: **10**' 'Answer: **332.74**'.",
    perturb_labels=None,
    include_options=False,
    chat_setup=True,
    batch_size=4,
    only_outputs=True,
    max_new_tokens=50
)


maths_0shot_value_df = evaluate_maths_answers(maths_0shot_value_df)

processing 0/3...
processing 1/3...
processing 2/3...


In [ ]:
maths_0shot_value_df.to_csv('maths_0shot_value.tsv', sep='\t', index=False)


In [ ]:
maths_0shot_value_df.head(20)

,prompt,response,correct_label,correct_value,distractor_label,response_clean,correct
0,Please answer the question. Format it as 'Answ...,Answer: **10**,D,3.00,F,10.0,False
1,Please answer the question. Format it as 'Answ...,Answer: **60**,D,45.00,F,60.0,False
2,Please answer the question. Format it as 'Answ...,Answer: **1**,A,0.00,F,1.0,False
3,Please answer the question. Format it as 'Answ...,Answer: **12**,B,7.00,F,12.0,False
4,Please answer the question. Format it as 'Answ...,Answer: **2**,D,0.95,F,2.0,False
5,Please answer the question. Format it as 'Answ...,Answer: **5**,B,6.00,F,5.0,False
6,Please answer the question. Format it as 'Answ...,Answer: **5**,C,3.00,F,5.0,False
7,Please answer the question. Format it as 'Answ...,Answer: **2**,A,2.00,F,2.0,True
8,Please answer the question. Format it as 'Answ...,Answer: **20**,C,3.00,F,20.0,False
9,Please answer the question. Format it as 'Answ...,Answer: **5**\n,D,8.00,F,5.0,False


In [ ]:
maths_0shot_label_df = run_experiment(
    llm=llm,
    test_examples=maths_subset,
    few_shot_examples=None,
    general_instruction="Please answer the question. Format it as 'Answer: ' and the correct answer option in parentheses, for example, 'Answer: (C)' or 'Answer: (A)'.",
    perturb_labels=None,
    include_options=True,
    chat_setup=True,
    batch_size=4,
    only_outputs=True,
    max_new_tokens=50
)

maths_0shot_label_df = evaluate_answer_labels(maths_0shot_label_df)
maths_0shot_label_df.to_csv('maths_0shot_label.tsv', sep='\t', index=False)


processing 0/3...
processing 1/3...
processing 2/3...


In [ ]:
maths_0shot_label_df.head(50)

#### Exercise 2.1. Answer the questions

1. Open the `.tsv` files for both experiments and calculate their accuracy (the percentage of correct answers). I suggest that you make the function for calculating accuracy for a dataFrame, so that you'd use it for the datasets below. In my subset, `maths_0shot_label` performs substantially better than `maths_0shot_value`. I would expect this to hold for any subset and, with enough examples (e.g., 100 or more), for its accuracy to approach a particular percentage. What is this percentage, and why? [2 points]
2. Examine both DataFrames in more detail. Do you notice any interesting patterns in the distributions of numerical answers or answer labels? How would you explain them? [2 point]

In [ ]:
# TODO-assignment: accuracy function for 2.1.1.

##### Chain-of-thought Reasoning

We have already discussed that we won't believe a human that would only give us a number to a complicated mathematical equation, even if the number was correct. We'd doubt even more, if we saw that this human shouted this number immediately after looking at the problem.

The way we've been prompting the model now is quite similar to that: the LLM's immediate next tokens were the answer: it's literally as if we made a person say the answer without giving them time on thinking!

Can we make it "fairer", or at least more efficient? One technique, suggested in [this paper](https://arxiv.org/pdf/2205.11916$0), is surprisingly simple yet effective: we just ask the model to think step by step before giving the final answer. This should allow it to unwrap the complex tasks, and (we hope) to apply mathematical reasoning to every step of the solution. That should imitate the expected behavior of a human who is solving a complex equation with a pen and paper before writing down the final answer. This approach is called **chain-of-thought** (usually abbreviated as **CoT**), and it proved to be very helpful in a number of NLP tasks related to reasoning.

Let's try this technique on maths! This will be done in the `maths_cot_df` experiment: you see, the `final_instruction` parameter includes the phrase "Think step by step, then provide the answer." In order to give enough space for model to "think", we also increased the `max_new_tokens` parameter.

In [ ]:
maths_cot_classic_df = run_experiment(
    llm=llm,
    test_examples=maths_subset,
    few_shot_examples=None,
    general_instruction="Please answer the question.",
    perturb_labels=None,
    include_options=False,
    final_instruction="Think step by step, then provide the answer. Format it as 'Answer: ' and the following number in bold font, for example, 'Answer: **10**' or 'Answer: **332.74**'.",
    chat_setup=True,
    batch_size=2,
    only_outputs=True,
    max_new_tokens=500
)


processing 0/3...
processing 1/3...
processing 2/3...


In [ ]:
maths_cot_classic_df = evaluate_maths_answers(maths_cot_classic_df)
maths_cot_classic_df.to_csv('maths_cot_classic.tsv', sep='\t', index=False)

In [ ]:
maths_cot_classic_df.head(20)

##### Post-Hoc Explainability

Let's think one more time about a human. A skilled one can calculate the answer in their head, give the answer, and explain it after giving the answer correctly.

Can we hope that the models also "think" somewhere inside themselves, and then only generate the answers? If this is true, then their explanations **after** giving answers should be reasonable, and their firstly uttered answers should be valid.

Is it true in reality? Let's run an experiment `maths_cot_posthoc`, where the only difference would be that we'd instruct the model to **first, give the answer, and then to explain it** (see `final_instruction`).

In [ ]:
maths_cot_posthoc_df = run_experiment(
    llm=llm,
    test_examples=maths_subset,
    few_shot_examples=None,
    general_instruction="Please answer the question.",
    perturb_labels=None,
    include_options=False,
    final_instruction="First, provide the answer, then explain it step by step. Format it as 'Answer: ' and the following number in bold font, for example, 'Answer: **10**' or 'Answer: **332.74**'.",
    chat_setup=True,
    batch_size=2,
    only_outputs=True,
    max_new_tokens=500
)


processing 0/3...
processing 1/3...
processing 2/3...


In [ ]:
maths_cot_posthoc_df = evaluate_maths_answers(maths_cot_posthoc_df)
maths_cot_posthoc_df.to_csv('maths_cot_posthoc.tsv', sep='\t', index=False)

In [ ]:
maths_cot_posthoc_df.head(20)

In [ ]:
maths_cot_classic_df.head(20)

#### Exercise 2.2. Answer the questions

1. Compare the `.tsv` datasets from the `maths_cot_classic` and `maths_cot_posthoc` experiments, and calculate their accuracy. Do their results differ? [1 point]
2. Manually compare the results of the exepriments `maths_cot_posthoc` with `maths_0shot_value`. Do you notice any similarities in the model's behavior? [1 point]

*Tip: All three datasets contain the same data points in the same order. You can combine them side by side and focus on cases where the model's answers differ, or use any other comparison method you prefer.*

In [ ]:
# TODO-assignment: accuracies for 2.2.1.

##### Using a Reasoning LLM

The success of chain-of-thought prompting inspired NLP researchers to improve model performance using similar techniques during both training and inference. This led to a class of models commonly called **reasoning LLMs**.

Although their architectures vary, one distinctive feature is the use of **thinking tokens**: tokens generated internally before the final answer, which may not be visible to the user. You may have encountered this in models such as DeepSeek or in ChatGPT's "thinking" mode (which displays these tokens if you press "Thought for X seconds").

Since we're working directly with LLM source code, we can inspect the thinking tokens! First, we'll examine what they look like, then compare a reasoning LLM's performance with that of a regular LLM.

You should already have a reasoning LLM loaded in the `reasoning_llm` variable. For this assignment, we'll use `Qwen/Qwen3-0.6B`.

Let's begin by explicitly asking the model to think step by step, using the same CoT prompt as we did for the regular LLM.

In [ ]:
del llm

In [ ]:
reasoning_llm = LLM("Qwen/Qwen3-0.6B", quantized=False, token_path=Path.home() / "hw2" / "credentials.json")


In [ ]:
reasoning_cot_df = run_experiment(
    llm=reasoning_llm,
    test_examples=maths_subset,
    few_shot_examples=None,
    general_instruction="Please answer the question.",
    perturb_labels=None,
    final_instruction="Think step by step, then provide the answer. Format it as 'Answer: ' and the following number in bold font, for example, 'Answer: **10**' or 'Answer: **332.74**'.",
    include_options=False,
    chat_setup=True,
    batch_size=1,
    only_outputs=True,
    max_new_tokens=2500
)


In [16]:
reasoning_cot_df = evaluate_maths_answers(reasoning_cot_df)
reasoning_cot_df.to_csv('reasoning_maths_cot_classic.tsv', sep='\t', index=False)

In [ ]:
reasoning_cot_df.head(20)

You can see that the model uses the html-like tags `<think>...</think>`. In the web UI of the models, the text between these sets of tokens is usually not shown. Yet, it is meant to simulate the "thinking" process of a human.

But how would it look like if we ask such a reasoning model to only give the answer? Let's call it `reasoning_cot_only_answer` experiment.

In [ ]:

reasoning_cot_only_answer_df = run_experiment(
    llm=reasoning_llm,
    test_examples=maths_subset,
    few_shot_examples=None,
    general_instruction="Return only the answer to the question. Format it as 'Answer: ' and the following number in bold font, for example, 'Answer: **10**' or 'Answer: **332.74**'.",
    perturb_labels=None,
    include_options=False,
    chat_setup=True,
    batch_size=1,
    only_outputs=True,
    max_new_tokens=2500
)


In [ ]:
reasoning_cot_only_answer_df = evaluate_maths_answers(reasoning_cot_only_answer_df)
reasoning_cot_only_answer_df.to_csv('reasoning_cot_only_answer.tsv', sep='\t', index=False)

#### Exercise 2.3. Answer the questions

1. Open the `.tsv` files for the `reasoning_cot` and `reasoning_cot_only_answer` experiments and calculate their accuracy. How much does performance differ? Do the exact instructions matter for a reasoning LLM if it generates "thinking tokens" anyway? [1 point]
2. Examine the generated explanations in the `.tsv` files in detail. Do you notice recurring phrases, constructions, or sequences? Are they all useful, or do some merely *appear* meaningful? Give one example of a recurring sequence that helps solve a math problem and another that adds little to the solution but appears frequently. [2 points]

In [ ]:
# TODO-assignment: accuracies for 2.3.1 (optionally for 2.3.2)

##### Reasoning models for factual answers

We've seen (probably on your data too), that the reasoning models show better performance on mathematical reasoning: despite uselessness of some of their frequent constructions, generally the inherent "thinking" path that they use before giving the answer helps them to break the question down into subproblems and resolve it better.

Should we now use them for any type of problem? Let's return to the subjects of your choice that you were using in Chapter 1. That would be our `reasoning_factual` experiment.

In [ ]:
reasoning_factual_df = run_experiment(
    llm=reasoning_llm,
    test_examples=subset['test'],
    few_shot_examples=perturb_random_4shot,
    perturb_labels=None,
    general_instruction="Please select the correct option to answer the question.",
    final_instruction="Return only the answer to the question. Format it as 'Answer: ' and the following letter in parentheses, for example, 'Answer: (D)', or 'Answer: (A)'.",
    include_options=True,
    chat_setup=True,
    batch_size=2,
    only_outputs=True,
    max_new_tokens=500
)

In [24]:
reasoning_factual_df = evaluate_maths_answers(reasoning_factual_df)
reasoning_factual_df.to_csv('reasoning_factual.tsv', sep='\t', index=False)

In [25]:
reasoning_factual_df.head()

,prompt,response,correct_label,correct_value,distractor_label,response_clean,correct
0,Please select the correct option to answer the...,\nAnswer: (B) (B) (B) (B (B) (B) (B) (B) (B) (...,D,Idols,F,None,False
1,Please select the correct option to answer the...,\nAnswer: (B)\nOkay\nOkay\nOkay\nOkay\nOkay\nO...,B,Yin-Yang School,F,None,False
2,Please select the correct option to answer the...,\nAnswer: (B)\nOkay\nOkay\nOkay\nOkay\n (B)\nO...,D,596 CE,F,None,False
3,Please select the correct option to answer the...,"\nOkay, the correct answer is (B) the followin...",A,Akitu,F,None,False
4,Please select the correct option to answer the...,\nAnswer: (B)\nOkay: (B)\nOkay: (B)\nOkay: (B)...,D,Ikebana,F,None,False


#### Exercise 2.4. Answer the questions:

1. Compare the accuracy of the regular LLM from Chapter 1 (`few_shot_{...}_comparison_None_results.tsv`, using the same `n` as above) with that of the reasoning LLM. You can also compare the answer-label distributions. Does either model clearly perform better on your subject? [1 point]
2. Examine `reasoning_factual.tsv` in detail. What recurring patterns do you notice in the model's phrases, sentences, or explanations? Are they genuinely useful for answering the questions, or do they merely *sound* reasonable without contributing much to the explanation or step-by-step reasoning? [2 points]

In [ ]:
# TODO-assignment: for 2.4.1. (optionally for 2.4.2)

##### Injection of Incorrect Answers

Finally, we'll test whether reasoning models can be misled when solving math problems. We'll use a method called **adversarial attack**: evaluating a model using deliberately manipulated inputs.

Here's how it works:

- When formatting a multiple-choice question, we'll wrap a **randomly chosen incorrect answer** in the HTML-like tag `<answer></answer>`.
- The model will be asked to output only the answer label. However, as a reasoning model, it will also generate thinking tokens along the way.
- We'll investigate two questions:
    - Is the model influenced by the incorrect HTML annotation? In other words, does it become more likely to select the annotated answer?
    - Does the annotation affect the model's reasoning? For example, does it explicitly notice that an answer has been tagged?

Let's test this using the `inject_reasoning` experiment.

In [26]:
inject_reasoning_df = run_experiment(
    llm=reasoning_llm,
    test_examples=maths_subset,
    few_shot_examples=None,
    general_instruction="Please answer the question.",
    perturb_labels='inject',
    final_instruction="Return only the answer to the question. Format it as 'Answer: ' and the following letter in parentheses, for example, 'Answer: (D)', or 'Answer: (A)'.",
    include_options=True,
    chat_setup=True,
    batch_size=2,
    only_outputs=True,
    max_new_tokens=500
)


[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


processing 0/3...


[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


processing 1/3...


[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


processing 2/3...


In [33]:
inject_reasoning_df = evaluate_answer_labels(inject_reasoning_df)
inject_reasoning_df.to_csv('inject_reasoning.tsv', sep='\t', index=False)

Now, let's visualize the distribution of the results. We will do that with the `visualize_injected_distribution` function, that would take on the dataset with an experiment output, and for each label (A-D), it will show the number of the correct answers, the misclassified LLM-generated answers, and among them, the answers that were matching an injected option.

In [ ]:
inject_reasoning_df.head(20)

In [28]:
def visualize_injected_distribution(df: pd.DataFrame):
    letters = ['A', 'B', 'C', 'D']
    x = np.arange(len(letters))
    width = 0.35

    correct_counts_list = []
    incorrect_matched_list = []
    incorrect_other_list = []

    for letter in letters:
        subset = df[df['response_clean'] == letter]

        # Correct answers (green segment)
        correct_count = subset['correct'].sum()

        # Incorrect answers matching distractor (blue with hatches)
        matched_count = ((subset['response_clean'] == subset['distractor_label']) &
                         (subset['distractor_label'] != 'F') &
                         (~subset['correct'])).sum()

        # Other incorrect answers (plain blue)
        other_count = len(subset) - correct_count - matched_count

        correct_counts_list.append(correct_count)
        incorrect_matched_list.append(matched_count)
        incorrect_other_list.append(other_count)

    correct_counts_list = np.array(correct_counts_list)
    incorrect_matched_list = np.array(incorrect_matched_list)
    incorrect_other_list = np.array(incorrect_other_list)

    # Ground Truth counts from correct_label column
    ground_truth_counts = df['correct_label'].value_counts().reindex(letters, fill_value=0)

    fig, ax = plt.subplots(figsize=(10, 7))

    # Model Outputs on the left side of the ticks
    ax.bar(x - width/2, correct_counts_list, width,
           label='Model Outputs (Correct)', color='#2ecc71', edgecolor='black')
    ax.bar(x - width/2, incorrect_matched_list, width, bottom=correct_counts_list,
           label='Model Outputs (Incorrect, Matches Injected)', color='#3498db', edgecolor='black', hatch='//')
    ax.bar(x - width/2, incorrect_other_list, width, bottom=correct_counts_list + incorrect_matched_list,
           label='Model Outputs (Incorrect, Other)', color='#aed6f1', edgecolor='black')

    # Ground Truth on the right side of the ticks
    ax.bar(x + width/2, ground_truth_counts, width,
           label='Ground Truth (correct_label)', color='#e74c3c', edgecolor='black')

    ax.set_ylabel('Counts')
    ax.set_title('Distribution of Model Predictions vs Ground Truth (Injected Label Analysis)')
    ax.set_xticks(x)
    ax.set_xticklabels(letters)
    ax.legend(loc='upper right')
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    fig.tight_layout()
    plt.show()

In [ ]:
visualize_injected_distribution(inject_reasoning_df)

Looks like the answer to our first question (are models influenced by adversarial HTML annotation) are true, especially for some classes...

#### Exercise 2.5. Answer the questions:

1. Open `inject_reasoning.tsv` and analyze the model's responses. Do models notice the injected answers and discuss them in their chain of thought? In particular, do they explicitly acknowledge the injected labels when selecting them as their final answers? [2 points]
2. You might argue that using `<answer>` tags is an artificial scenario unlikely to occur in practice. Can you think of any real-world situations where a similar problem might arise? [1 point]

## Submission guidelines:

1. You need to **submit two files** - the modified `.ipynb` with the filled fields where necessary, and the `.pdf` report with the detailed answers to your questions.
2. The total sum of the answers is **28**:
    - Exercise 1.1.: 5
    - Exercise 1.2.: 9
    - Exercise 2.1.: 3
    - Exercise 2.2.: 2
    - Exercise 2.3.: 3
    - Exercise 2.4.: 3
    - Exercise 2.5.: 3

 Just as with the previous assignment, you don't have to do all exercises; 25 is the maximal number of the points for the assignment and you can subselect the tasks to reach that score. Any points that exceed that will be allocated to your next assignments.

3. Don't hesitate to ask questions, especially when they are about the organizational matters (errors when importing libraries/running GPU)! However, please **first check the Forum page at the OLAT**, to see if there already were solutions to the same problems, and don't hesitate to send your suggested solutions!